In [1]:
import sys
sys.path.insert(0,'/mnt/AEA8F340A8F3059D/sportsbet/ai-engine')

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from tools.mongodbtools import fetchMatch,fetchTeam,fetchMatchOdds
from rag.retriever import retrieveChunks
from config.settings import settings
import json,logging

In [3]:
logger=logging.getLogger(__name__)

In [4]:
MATCH_ANALYSIS_PROMPT="""You are an expert cricket analyst for a sports betting platform.

Generate a comprehensive pre-match analysis based on the data provided.

MATCH DATA:
{match_data}

HEAD-TO-HEAD HISTORY:
{h2h_data}

CURRENT ODDS:
{odds_data}

TEAM DETAILS:
{team_data}

RELEVANT NEWS & CONTEXT:
{rag_context}

USER CONTEXT:
{user_context}

Generate a detailed analysis as JSON with this exact structure:
{{
  "summary": "2-3 sentence executive summary",
  "teamAnalysis": {{
    "team1": {{
      "name": "team name",
      "strengths": ["list of strengths"],
      "weaknesses": ["list of weaknesses"],
      "keyPlayers": ["player names"],
      "recentForm": "description of recent form"
    }},
    "team2": {{
      "name": "team name",
      "strengths": ["..."],
      "weaknesses": ["..."],
      "keyPlayers": ["..."],
      "recentForm": "..."
    }}
  }},
  "h2hAnalysis": {{
    "totalMatches": 0,
    "team1Wins": 0,
    "team2Wins": 0,
    "insight": "key takeaway from H2H"
  }},
  "conditions": {{
    "venue": "venue name if known",
    "pitch": "pitch analysis if available",
    "weather": "weather info if available",
    "impact": "how conditions might affect the match"
  }},
  "bettingRecommendation": {{
    "favored": "team name",
    "confidence": 0.0-1.0,
    "reasoning": "why this team is favored",
    "riskLevel": "low|medium|high",
    "suggestedBet": "specific betting suggestion"
  }}
}}
"""

In [6]:
async def matchAnalyzer(state):
    ctx=state['context']
    matchId=ctx['matchId'] or state.get('slots',{}).get("matchId","")
    matchData=state.get("match-data",{})
    if not matchData and matchId:
        matchData=await fetchMatch(matchId) or {}
    oddsData={}
    if matchId:
        oddsData=await fetchMatchOdds(matchId) or {}
    teamData=[]
    for tid in teamIds[:2]:
        team=await fetchTeam(str(tid))
        if team:
            teamData.append(team)
    
    matchTitle=matchData.get("title","cricket match")
    ragChunks=state.get("rag-chunks",[])
    if not ragChunks:
        ragChunks=await retrieveChunks(matchTitle,topK=5)
    llm=ChatGoogleGenerativeAI(
        model=settings.GEMINI_MODEL,
        google_api_key=settings.GEMINI_API_KEY,
        temperature=0.3,
    )
    prompt=ChatPromptTemplate.from_messages([
        ("system",MATCH_ANALYSIS_PROMPT),
    ])
    try:
        result=await (prompt|llm).ainvoke({
            "match-data":json.dumps(matchData,default=str)[:1000],
            "odds-data":json.dumps(oddsData,default=str)[:1000],
            "team-data":json.dumps(teamData,default=str)[:1000],
            "rag-context":json.dumps(ragChunks,default=str)[:1000],
            "user-context":json.dumps({"recentBets":state.get("user-recent-bets",[])[:5],
            "wallet":state.get("user-wallet",{})
            },default=str)[:100]
        })
        content=result.content.strip()
        if "```" in content:
            content=content.split("```")[1].split("```")[0]
            if content.startswith("json"):
                content=content[4:]
        output=json.loads(content)
    except Exception as e:
        logger.error(f"match analyzer fail {e}")
        output={
            "summary":f"Analysis for {matchTitle}",
            "error":str(e)
        }
    return{
        "output":output,
        "match-data":matchData,
    }